# Gold layer: Brand Manager, questions 3 and 4

This notebook answers:

- Q3: Which brand has the largest share of sales within its category, and is that share stable over time?
- Q4: Which products have an actual sale price that differs from the retail price by more than 15%, and are these cases concentrated in a particular brand or category?

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

SOURCE = "workspace.brand_silver"
GOLD = "workspace.brand_gold"

brand_revenue = spark.table(f"{GOLD}.brand_revenue")
lineitem = spark.table(f"{SOURCE}.lineitem")
part = spark.table(f"{SOURCE}.part")

In [0]:
display(brand_revenue.limit(10))

## Q3. Largest brand share within category and stability over time

Market share is calculated as brand revenue divided by total revenue of its category (`p_mfgr`).

The leading brand is identified within each category. The brand with the highest share among these category leaders is then analyzed quarterly using standard deviation and range.

In [0]:
# Overall market share by brand
w_category = Window.partitionBy("p_mfgr")

brand_share = (
    brand_revenue
    .groupBy("p_mfgr", "p_brand")
    .agg(F.sum("revenue").alias("revenue"))
    .withColumn(
        "share_pct",
        F.round(100 * F.col("revenue") / F.sum("revenue").over(w_category), 2)
    )
)

# Brand with the largest share within its category
top_brand = brand_share.orderBy(F.desc("share_pct")).first()

top_mfgr = top_brand["p_mfgr"]
top_brand_name = top_brand["p_brand"]

print(f"Top brand: {top_brand_name}")
print(f"Category: {top_mfgr}")
print(f"Overall share: {top_brand['share_pct']}%")

display(
    brand_share
    .filter(F.col("p_mfgr") == top_mfgr)
    .orderBy(F.desc("share_pct"))
)

In [0]:
# Brand with the largest market share in each category
rank_window = Window.partitionBy("p_mfgr").orderBy(F.desc("share_pct"))

category_winners = (
    brand_share
    .withColumn("rank", F.row_number().over(rank_window))
    .filter(F.col("rank") == 1)
    .drop("rank")
    .orderBy("p_mfgr")
)

display(category_winners)

In [0]:
# Quarterly market share of all brands
w_quarter = Window.partitionBy(
    "p_mfgr",
    "order_year",
    "order_quarter"
)

quarterly_share = (
    brand_revenue
    .withColumn(
        "category_revenue",
        F.sum("revenue").over(w_quarter)
    )
    .withColumn(
        "share_pct",
        F.round(
            100 * F.col("revenue") / F.col("category_revenue"),
            2
        )
    )
)

# Save quarterly market share to Gold
quarterly_share.write.mode("overwrite").saveAsTable(
    f"{GOLD}.brand_market_share"
)

# Read the Gold table back
quarterly_share = spark.table(
    f"{GOLD}.brand_market_share"
)

In [0]:
# Share dynamics of the top brand
top_brand_history = (
    quarterly_share
    .filter(
        (F.col("p_mfgr") == top_mfgr) &
        (F.col("p_brand") == top_brand_name)
    )
    .select(
        "order_year",
        "order_quarter",
        "revenue",
        "share_pct"
    )
    .orderBy("order_year", "order_quarter")
)

display(top_brand_history)

In [0]:
# Stability of the top brand's quarterly market share
stability = (
    top_brand_history
    .agg(
        F.avg("share_pct").alias("avg_share_pct"),
        F.stddev_samp("share_pct").alias("stddev_share_pct"),
        F.min("share_pct").alias("min_share_pct"),
        F.max("share_pct").alias("max_share_pct"),
    )
    .withColumn(
        "range_pp",
        F.col("max_share_pct") - F.col("min_share_pct")
    )
)

display(stability)

### Q3 conclusion

The brand with the highest overall market share is **Brand#33** in **Manufacturer#3**, with a share of **20.20%**.

Its quarterly market share is highly stable over time. The average quarterly share is **20.21%**, ranging only from **20.00% to 20.37%**. The range is **0.37 percentage points**, and the standard deviation is approximately **0.09 percentage points**.

Therefore, Brand#33 maintains a very stable position within its category, with no significant changes in market share across the observed quarters.

## Q4. Actual sale price vs retail price

Actual unit sale price is calculated as net revenue after discount divided by quantity.

Products with a price difference greater than 15% from `p_retailprice` are identified and analyzed by brand and category.

In [0]:
# Actual sale price aggregated by product
product_price_analysis = (
    lineitem
    .join(part, lineitem.l_partkey == part.p_partkey)
    .groupBy(
        "p_partkey",
        "p_name",
        "p_mfgr",
        "p_brand",
        "p_retailprice"
    )
    .agg(
        (
            F.sum(F.col("l_extendedprice") * (1 - F.col("l_discount")))
            / F.sum("l_quantity")
        ).alias("actual_unit_price"),
        F.sum("l_quantity").alias("quantity_sold"),
        F.count("*").alias("line_items")
    )
    .withColumn(
        "price_diff_pct",
        100 * F.abs(F.col("actual_unit_price") - F.col("p_retailprice"))
        / F.col("p_retailprice")
    )
)

# Save product-level analysis to Gold
product_price_analysis.write.mode("overwrite").saveAsTable(
    f"{GOLD}.product_price_analysis"
)

product_price_analysis = spark.table(
    f"{GOLD}.product_price_analysis"
)

# Check observed product-level price differences
price_stats = product_price_analysis.agg(
    F.round(F.min("price_diff_pct"), 2).alias("min_diff_pct"),
    F.round(F.avg("price_diff_pct"), 2).alias("avg_diff_pct"),
    F.round(F.max("price_diff_pct"), 2).alias("max_diff_pct"),
)

display(price_stats)

# Products with more than 15% difference
price_outliers = product_price_analysis.filter(
    F.col("price_diff_pct") > 15
)

q4_summary = price_outliers.agg(
    F.count("*").alias("affected_products")
)

display(q4_summary)

In [0]:
affected_products = price_outliers.count()

if affected_products > 0:
    print("Concentration by brand:")
    display(
        price_outliers
        .groupBy("p_brand")
        .agg(F.count("*").alias("affected_products"))
        .orderBy(F.desc("affected_products"))
    )

    print("Concentration by category:")
    display(
        price_outliers
        .groupBy("p_mfgr")
        .agg(F.count("*").alias("affected_products"))
        .orderBy(F.desc("affected_products"))
    )
else:
    print("No products differ from retail price by more than 15%.")
    print("Therefore, there is no concentration by brand or category.")

### Q4 conclusion

No products have an actual unit sale price that differs from the retail price by more than **15%**.

The maximum observed product-level price difference is below the 15% threshold. Therefore, the number of affected products is **0**, and there is no concentration within any particular brand or category.

## Validation

In [0]:
# V1. Market shares must sum to 100% within each category and quarter
share_check = (
    quarterly_share
    .groupBy("p_mfgr", "order_year", "order_quarter")
    .agg(F.sum("share_pct").alias("total_share_pct"))
)

max_share_error = (
    share_check
    .select(F.max(F.abs(F.col("total_share_pct") - 100)))
    .first()[0]
)

print(f"Maximum market-share sum error: {max_share_error}")

assert max_share_error <= 0.1, "Market shares do not sum to approximately 100%"
print("V1 passed")

In [0]:
# V2. Product price Gold table contains exactly one row per product
rows = product_price_analysis.count()
unique_products = product_price_analysis.select("p_partkey").distinct().count()

print(f"Rows: {rows:,}")
print(f"Unique products: {unique_products:,}")

assert rows == unique_products, "Products are duplicated in Gold price analysis"
print("V2 passed")

### Validation conclusions

**V1 — Market share consistency**

For each manufacturer and quarter, brand market shares sum to approximately 100%.  
The maximum observed deviation from 100% is **0.01 percentage points**, which is caused by rounding individual brand shares to two decimal places.

Therefore, the calculated market shares are internally consistent.

**V2 — Product-level uniqueness**

The `product_price_analysis` Gold table contains **1,000,000 rows** and **1,000,000 unique products**.

Therefore, each product appears exactly once in the product-level Gold table, confirming that the aggregation does not introduce duplicate product records.